<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m4_03_regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M4.03 — Keeping the weights under control

In the lab we saw that a linear model with many features can overfit,
and that **regularisation** (`Ridge` and its `alpha`, the `C` of the
logistic regression) holds its weights back. In this workbook you
regularise **the traffic model**, in a situation where it badly needs
it: little data and hundreds of useless columns. You will watch the
weights shrink as `alpha` grows, compare two kinds of penalty, and see
two things that go wrong without regularisation or without scaling.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "depends_on_units": "6d9cd7b2184f2d4f",
    "huge_alpha_prediction": "4fc137f367c48618",
    "junk_effect_on_testing_error": "e5ff8d2b6809d810",
    "junk_effect_on_training_error": "8c6fb1e9e37a1aea",
    "regime_C_100": "a1f32f577f69843e",
    "regime_largest_alpha": "5a43f1e23e0f76ff",
    "regime_smallest_alpha": "a1f32f577f69843e",
    "ridge_exact_zeros": "5feceb66ffc86f38",
    "selects_features": "fad4382334c0bcf6",
    "stable_weights": "409e364456fa9364",
    "strongest_regularisation": "2d9c2244d7dad7fb"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. One month of data and 300 useless columns

A new traffic sensor has been running for **one month** (January 2016).
We must build a model from that month and it will be judged on the seven
months that follow.

We describe each hour with the features that worked in workbook M4.02
(one column per hour of the day, the workday, and their interactions),
plus four weather measurements: 53 **real** columns.

And there is a complication. A colleague, "to be on the safe side", has
attached **300 more columns** taken from sensors that have nothing to do
with the road. We simulate them with random numbers and call them
`junk_000` to `junk_299`. Nothing in them can help.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

traffic = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/metro_traffic.csv",
    parse_dates=["date_time"],
)
traffic = traffic[traffic["rain_1h"] < 100].reset_index(drop=True)  # the impossible reading of M4.01
traffic["hour"] = traffic["date_time"].dt.hour
traffic["temp_c"] = traffic["temp"] - 273.15
traffic["workday"] = (
    (traffic["date_time"].dt.day_of_week < 5) & (traffic["holiday"] == "No holiday")
).astype(int)


def describe(rows, n_junk=300):
    """The 53 real columns, followed by `n_junk` columns of random numbers."""
    columns = {}
    for hour in range(24):
        columns[f"hour_{hour:02d}"] = (rows["hour"] == hour).astype(float)
        columns[f"hour_{hour:02d}_on_workday"] = ((rows["hour"] == hour) & (rows["workday"] == 1)).astype(float)
    columns["workday"] = rows["workday"].astype(float)
    for name in ["temp_c", "clouds_all", "rain_1h", "snow_1h"]:
        columns[name] = rows[name]
    junk = np.random.default_rng(0).normal(size=(len(rows), n_junk))
    for i in range(n_junk):
        columns[f"junk_{i:03d}"] = junk[:, i]
    return pd.DataFrame(columns, index=rows.index)


data = describe(traffic)
target = traffic["traffic_volume"]
real_columns = [name for name in data.columns if not name.startswith("junk")]
junk_columns = [name for name in data.columns if name.startswith("junk")]

is_train = traffic["date_time"] < "2016-02-01"
is_test = (traffic["date_time"] >= "2016-02-01") & (traffic["date_time"] < "2016-09-01")
data_train, target_train = data[is_train], target[is_train]
data_test, target_test = data[is_test], target[is_test]
print(f"{len(real_columns)} real columns + {len(junk_columns)} junk columns = {data.shape[1]} columns")
print(f"{len(data_train)} hours to learn from, {len(data_test)} hours to test on")

## 2. A linear regression drowning in columns

We train an ordinary linear regression twice: once on the 53 real
columns only, once on all 353 columns.

🔮 **Predict first.** The junk columns are pure noise. Compared with the
model that only has the real columns, the model given all the columns
will have a **training** error that is… `"lower"`, `"the same"` or
`"higher"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["lower", "the same", "higher"])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

results = {}


def evaluate(name, model, columns=None):
    """Train on January, report training and testing errors."""
    columns = list(data.columns) if columns is None else columns
    model.fit(data_train[columns], target_train)
    results[name] = {
        "training error": mean_absolute_error(target_train, model.predict(data_train[columns])),
        "testing error": mean_absolute_error(target_test, model.predict(data_test[columns])),
    }
    return model


evaluate("linear regression, real columns only", make_pipeline(StandardScaler(), LinearRegression()), real_columns)
evaluate("linear regression, all columns", make_pipeline(StandardScaler(), LinearRegression()))
pd.DataFrame(results).T.round(0)

✏️ What did the 300 junk columns do to the **training** error and to the
**testing** error? Write `"lower"` or `"higher"` for each.

In [ ]:
junk_effect_on_training_error = ____
junk_effect_on_testing_error = ____
check("junk_effect_on_training_error", junk_effect_on_training_error)
check("junk_effect_on_testing_error", junk_effect_on_testing_error)

With 353 weights to choose and only 461 hours of data, the model uses
the random columns to fit the accidents of January. Each junk column is
useless on its own, but 300 of them together can "explain" a lot of
noise. This is overfitting, obtained without any polynomial, just by
adding columns.

## 3. Ridge: paying for large weights

`Ridge` is the same linear model with one extra rule: **large weights
cost something**. While learning, it balances two goals: fit the
training data, and keep the weights small. `alpha` sets the price of the
weights: with `alpha` near 0 it behaves like the ordinary linear
regression, and the larger `alpha`, the smaller the weights.

In [ ]:
from sklearn.linear_model import Ridge

alphas = np.logspace(-1, 5, 13)
rows, ridge_weights = [], []
for alpha in alphas:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(data_train, target_train)
    ridge_weights.append(ridge[-1].coef_)
    rows.append(
        {
            "alpha": alpha,
            "training error": mean_absolute_error(target_train, ridge.predict(data_train)),
            "testing error": mean_absolute_error(target_test, ridge.predict(data_test)),
        }
    )
ridge_curve = pd.DataFrame(rows).set_index("alpha")
ridge_weights = pd.DataFrame(ridge_weights, index=alphas, columns=data.columns)

ax = ridge_curve.plot(logx=True, marker="o", figsize=(7, 3.5))
_ = ax.set(xlabel="alpha (log scale)", ylabel="mean absolute error", title="Validation curve of Ridge")

This is a validation curve, as in module 2, and this time it has the
textbook U shape.

✏️ Name the regime at each end. Write `"underfitting"` or
`"overfitting"`.

In [ ]:
regime_smallest_alpha = ____
regime_largest_alpha = ____
check("regime_smallest_alpha", regime_smallest_alpha)
check("regime_largest_alpha", regime_largest_alpha)

### ➕ Going further: watching the weights shrink

Each line below is the weight of **one column** as `alpha` grows: the 53
real columns in colour, the 300 junk columns in grey.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas, ridge_weights[junk_columns], color="lightgray", linewidth=0.5)
ax.plot(alphas, ridge_weights[real_columns], linewidth=1)
ax.set(xscale="log", xlabel="alpha (log scale)", ylabel="weight", title="Weights of the Ridge model")
_ = ax.axhline(0, color="black", linewidth=0.5)

✏️ With a very large `alpha`, all the weights are practically zero. What
does the model then predict for **every** hour? Write `"zero"`, `"the
mean of the training target"` or `"the last value seen"`.

In [ ]:
huge_alpha_prediction = ____
check("huge_alpha_prediction", huge_alpha_prediction, hint="The weights are near zero, but the intercept is not penalised.")

So the two ends of the curve are two models you already know: the
ordinary linear regression on the left, the baseline that always
predicts the mean on the right. `alpha` is a dial between them.

scikit-learn's example
[Ridge coefficients as a function of the regularization](https://scikit-learn.org/1.8/auto_examples/linear_model/plot_ridge_coeffs.html)
draws the same picture on artificial data.

🧠 **Think.** Ridge shrinks the junk weights, but it shrinks the real
ones too. At its best `alpha`, is Ridge better or worse than the linear
regression that was given **only the real columns** in section 2? What
does that tell you about regularisation as a replacement for choosing
good features?

## 4. ➕ Going further: a penalty that switches columns off

Ridge makes weights small. It never makes them **exactly zero**: every
junk column keeps a little influence. `Lasso` is a linear model with a
different penalty (called L1; Ridge's is called L2), and its penalty has
a remarkable property.

🔮 **Predict first.** Out of the 300 junk columns, how many will the
Lasso give a weight of **exactly 0**? **"none"**, **"about half"** or
**"almost all"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["none", "about half", "almost all"])

In [ ]:
from sklearn.linear_model import Lasso

best_alpha = ridge_curve["testing error"].idxmin()
ridge = evaluate("Ridge", make_pipeline(StandardScaler(), Ridge(alpha=best_alpha)))
lasso = evaluate("Lasso", make_pipeline(StandardScaler(), Lasso(alpha=30, max_iter=50_000)))
weights = pd.DataFrame({"Ridge": ridge[-1].coef_, "Lasso": lasso[-1].coef_}, index=data.columns)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, name in zip(axes, weights.columns):
    ax.bar(range(len(weights)), weights[name], width=1.0)
    ax.axvline(len(real_columns) - 0.5, color="red", linestyle="--")
    ax.set(title=f"Weights of the {name} model", xlabel="column (real columns left of the red line, junk right)")
axes[0].set_ylabel("weight")

exact_zeros = pd.DataFrame(
    {
        "junk columns set to exactly 0": (weights.loc[junk_columns] == 0).sum(),
        "real columns set to exactly 0": (weights.loc[real_columns] == 0).sum(),
    }
)
exact_zeros

✏️ How many junk columns did **Ridge** set to exactly 0? And which model
can be used to **select** features? Write `"Ridge"` or `"Lasso"`.

In [ ]:
ridge_exact_zeros = ____
selects_features = ____
check("ridge_exact_zeros", ridge_exact_zeros)
check("selects_features", selects_features)

In [ ]:
pd.DataFrame(results).T.round(0)

The Lasso has ignored most of the junk by itself and comes close to the
model that was given only the real columns. A column with a weight of
exactly zero can be removed from the data: the Lasso is a model and a
**feature selector** at once.

🧠 **Think.** The Lasso also switched off some **real** columns. Look at
which ones (`weights.loc[real_columns]`). Is that necessarily a mistake?

## 5. ➕ Going further: two thermometers

Back to the full two years of data and a different danger. The table has
the temperature in degrees Celsius. Suppose a second sensor reports the
same temperature in **Fahrenheit**, rounded to the degree. The two
columns say almost exactly the same thing: they are **collinear**.

We train a model on these two columns alone, five times, each time
leaving out a different fifth of the data, and look at the two weights.

In [ ]:
from sklearn.model_selection import KFold, cross_validate

two_thermometers = pd.DataFrame(
    {"temp_c": traffic["temp_c"], "temp_f": (traffic["temp_c"] * 9 / 5 + 32).round(0)}
)
folds = KFold(n_splits=5, shuffle=True, random_state=0)


def weights_per_fold(model):
    fitted = cross_validate(model, two_thermometers, target, cv=folds, return_estimator=True)["estimator"]
    return pd.DataFrame(
        [estimator[-1].coef_ for estimator in fitted],
        columns=["weight of temp_c", "weight of temp_f"],
        index=pd.RangeIndex(1, 6, name="fold"),
    )


ordinary_weights = weights_per_fold(make_pipeline(StandardScaler(), LinearRegression()))
ridge_weights_folds = weights_per_fold(make_pipeline(StandardScaler(), Ridge(alpha=100)))
pd.concat({"linear regression": ordinary_weights, "Ridge": ridge_weights_folds}, axis=1).round(0)

Both columns measure the temperature, and we know from workbook M4.01
that warmer hours have a little more traffic. Read the weights of the
ordinary linear regression fold by fold: in some folds `temp_c` gets a
large positive weight and `temp_f` a negative one, in others the
reverse.

✏️ With which model do the weights keep the same sign and about the same
size in every fold? Write `"linear regression"` or `"Ridge"`.

In [ ]:
stable_weights = ____
check("stable_weights", stable_weights)

When two columns carry the same information, only their **combined**
effect is fixed by the data. The linear regression is free to give
+1000 to one and −900 to the other, or the opposite, and tiny changes in
the data decide which. The predictions are fine; the **weights mean
nothing**. Ridge prefers small weights, so it splits the effect between
the two columns, the same way every time.

🧠 **Think.** Somebody reads the first fold and concludes: "the hotter
it is in Fahrenheit, the less traffic". What do you answer?

## 6. ➕ Going further: the penalty depends on the units

Ridge makes large weights expensive. But the size of a weight depends on
the **unit** of its feature. Cloud cover is stored in percent (0 to
100). Stored as a fraction (0 to 1), the same model needs a weight 100
times larger.

🔮 **Predict first.** We change the cloud cover from percent to fraction
and retrain. For which model will the **predictions change**? `"linear
regression"`, `"Ridge without scaling"` or `"Ridge after a
StandardScaler"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["linear regression", "Ridge without scaling", "Ridge after a StandardScaler"])

In [ ]:
in_percent = traffic[["temp_c", "clouds_all", "hour"]]
in_fraction = in_percent.assign(clouds_all=in_percent["clouds_all"] / 100)
candidates = {
    "linear regression": LinearRegression(),
    "Ridge without scaling": Ridge(alpha=10_000),
    "Ridge after a StandardScaler": make_pipeline(StandardScaler(), Ridge(alpha=10_000)),
}
change = {}
for name, model in candidates.items():
    predictions_percent = model.fit(in_percent, target).predict(in_percent)
    predictions_fraction = model.fit(in_fraction, target).predict(in_fraction)
    change[name] = np.abs(predictions_percent - predictions_fraction).max()
pd.Series(change, name="largest change in a prediction (vehicles per hour)").round(1).to_frame()

✏️ Which model gives different predictions when a unit changes?

In [ ]:
depends_on_units = ____
check("depends_on_units", depends_on_units)

A model whose predictions depend on whether somebody wrote 0.4 or 40% is
not acceptable. This is the third reason to scale before a linear model
(after convergence in module 1 and comparable weights in workbook
M4.01): **regularisation only treats the features fairly when they are
on the same scale**.

## 7. The same idea for classification: `C`

`LogisticRegression` is regularised too, by default. Its dial is called
`C` and it works **in the opposite direction** from `alpha`: `C` is the
freedom given to the weights, not their price.

We predict heavy traffic (more than 4500 vehicles) from the January data
with all 353 columns, for several values of `C`.

In [ ]:
from sklearn.linear_model import LogisticRegression

heavy_train, heavy_test = target_train > 4500, target_test > 4500
rows = []
for C in [0.0001, 0.01, 1, 100]:
    classifier = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
    classifier.fit(data_train, heavy_train)
    rows.append(
        {
            "C": C,
            "size of the weights": np.linalg.norm(classifier[-1].coef_),
            "training accuracy": classifier.score(data_train, heavy_train),
            "testing accuracy": classifier.score(data_test, heavy_test),
        }
    )
by_C = pd.DataFrame(rows).set_index("C")
by_C.round(3)

✏️ Which value of `C` regularises the **most** (the smallest weights):
the smallest or the largest? And is the model with `C=100` underfitting
or overfitting?

In [ ]:
strongest_regularisation = ____
regime_C_100 = ____
check("strongest_regularisation", strongest_regularisation, hint='Write "smallest C" or "largest C".')
check("regime_C_100", regime_C_100)

🧠 **Think.**

1. `Ridge(alpha=0.001)` and `LogisticRegression(C=0.001)`: which of the
   two is almost unregularised, and which is strongly regularised?
2. `LogisticRegression()` uses `C=1` unless you say otherwise. In module
   1 you used it many times without knowing it was regularised. Does
   that change how you read the weights of those models?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - what the 300 junk columns did to the linear regression, and the
     validation curve of Ridge;
   - the picture of the shrinking weights, and what the model predicts
     at each end of the `alpha` dial;
   - Ridge against Lasso: which one switches columns off, and why that
     is useful;
   - the two thermometers: why the weights flip sign and what Ridge
     changes;
   - why the predictions of Ridge changed when cloud cover went from
     percent to fraction.